[Competition Link](https://www.kaggle.com/competitions/titanic/overview)

---



In [4]:
from google.colab import files
files.download('/content/titanic.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import shutil

shutil.unpack_archive('/content/titanic.zip', '/content/titanic')

---

In [46]:
# Imports
from pathlib import Path
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score

In [47]:
# Import the data
train_df = pd.read_csv('/content/titanic/train.csv')
test_df = pd.read_csv('/content/titanic/test.csv')

print("-" * 40)

print(f"| Train shape: {train_df.shape}")
print(f"| Test shape: {test_df.shape}")

print("-" * 40)

print(train_df.head(), "\n")
print(test_df.head(), "\n")

----------------------------------------
| Train shape: (891, 12)
| Test shape: (418, 11)
----------------------------------------
   PassengerId  Survived  Pclass  ...     Fare Cabin  Embarked
0            1         0       3  ...   7.2500   NaN         S
1            2         1       1  ...  71.2833   C85         C
2            3         1       3  ...   7.9250   NaN         S
3            4         1       1  ...  53.1000  C123         S
4            5         0       3  ...   8.0500   NaN         S

[5 rows x 12 columns] 

   PassengerId  Pclass  ... Cabin Embarked
0          892       3  ...   NaN        Q
1          893       3  ...   NaN        S
2          894       2  ...   NaN        Q
3          895       3  ...   NaN        S
4          896       3  ...   NaN        S

[5 rows x 11 columns] 



In [48]:
# Quick survival insights
print("-" * 40)
print(f"|-- OVERALL SURVIVAL RATE:", round(train_df['Survived'].mean(), 3))
print(f"\n|-- SURVIVAL BY SEX:\n", train_df.groupby("Sex")["Survived"].mean().sort_values(ascending=False))
print(f"\n|-- SURVIVAL BY PClASS:\n", train_df.groupby("Pclass")["Survived"].mean().sort_values(ascending=False))

train_df["AgeBand"] = pd.cut(train_df["Age"], bins=[0, 12, 18, 35, 60, 100])
print(f"\n|-- SURVIVAL BY AGEBAND:\n", train_df.groupby("AgeBand")["Survived"].mean().sort_values(ascending=False))

print("-" * 40)

----------------------------------------
|-- OVERALL SURVIVAL RATE: 0.384

|-- SURVIVAL BY SEX:
 Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

|-- SURVIVAL BY PClASS:
 Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

|-- SURVIVAL BY AGEBAND:
 AgeBand
(0, 12]      0.579710
(12, 18]     0.428571
(35, 60]     0.400000
(18, 35]     0.382682
(60, 100]    0.227273
Name: Survived, dtype: float64
----------------------------------------


/tmp/ipykernel_18718/1265921577.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(f"\n|-- SURVIVAL BY AGEBAND:\n", train_df.groupby("AgeBand")["Survived"].mean().sort_values(ascending=False))


In [ ]:
# Feature engineering Submission 1: Score = 0.76794
def add_features(df):
  out = df.copy()
  
  # Title from name (e.g. Mr., Miss., etc.)
  out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).fillna("Unknown")
  
  # Family size and whether alone
  out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
  out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
  
  # Keep deck letter only
  out["Deck"] = out["Cabin"].fillna("Unknown").astype(str).str[0]
  
  return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)

target = "Survived"
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin", "AgeBand"] # AgeBand from EDA, not in test set

x = train_fe.drop(columns=[target] + [c for c in drop_cols if c in train_fe.columns])
y = train_fe[target]

x_test = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

cat_cols = x.select_dtypes(include=["object"]).columns.tolist()
num_cols = x.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

Categorical columns: ['Sex', 'Embarked', 'Title', 'Deck']
Numerical columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']


In [ ]:
# Feature engineering Submission 2: Score = 0.72727
def add_features(df):
  out = df.copy()
  
  out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).fillna("Unknown")
  # [UPDATE FROM] submission 1: Map rare titles to "Rare" and standardize some titles
  title_map = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Capt": "Rare",
    "Col": "Rare",
    "Don": "Rare",
    "Dr": "Rare",
    "Major": "Rare",
    "Rev": "Rare",
    "Sir": "Rare",
    "Jonkheer": "Rare",
    "Dona": "Rare"
  }
  
  # [UPDATE FROM] submission 1: Map rare titles to "Rare" and standardize some titles
  out["Title"] = out["Title"].replace(title_map)
  out["Title"] = out["Title"].where(out["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
  
  # [UPDATE FROM] submission 1: Add family size bands 
  out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
  out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
  out["FamilySizeBand"] = pd.cut(out["FamilySize"], bins=[0, 1, 4, 20], labels=["Alone", "Small", "Large"])
  
  # [UPDATE FROM] submission 1: Keep deck letter only, but also mark missing as "Unknown"
  out["Deck"] = out["Cabin"].fillna("Unknown").astype(str).str[0]
  out["TicketPrefix"] = (
    out["Ticket"]
    .astype(str)
    .str.replace(r"[./]", " ", regex=True)
    .str.replace(r"\d+", "", regex=True)
    .str.strip()
    .replace("", "Unknown")
  )
  
  # [UPDATE FROM] submission 1: Fill missing embarked with mode, fare with median, and add log fare
  out["Embarked"] = out["Embarked"].fillna(out["Embarked"].mode()[0])
  out["Fare"] = out["Fare"].fillna(out["Fare"].median())
  out["FareLog"] = np.log1p(out["Fare"])
  
  return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)

target = "Survived"
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin", "AgeBand"] # AgeBand from EDA, not in test set

x = train_fe.drop(columns=[target] + [c for c in drop_cols if c in train_fe.columns])
y = train_fe[target]
x_test = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

# [UPDATE FROM] submission 1: Fill missing age with median age
age_fill = train_fe.groupby(["Sex", "Pclass", "Title"])["Age"].median()

# [UPDATE FROM] submission 1: Create a function to fill age based on median age of groups
def fill_age(df):
  out = df.copy()
  out["Age"] = out["Age"].fillna(
    out.groupby(["Sex", "Pclass", "Title"])["Age"].transform("median")
  )
  return out

# [UPDATE FROM] submission 1: Fill missing age in both train and test sets
x = fill_age(x)
x_test = fill_age(x_test)

cat_cols = x.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = x.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

Categorical columns: ['Sex', 'Embarked', 'Title', 'FamilySizeBand', 'Deck', 'TicketPrefix']
Numerical columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FareLog']


In [49]:
# Feature engineering Submission 3: Score =

def add_features(df):
  out = df.copy()

  out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).fillna("Unknown")
  title_map = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Capt": "Rare",
    "Col": "Rare",
    "Don": "Rare",
    "Dr": "Rare",
    "Major": "Rare",
    "Rev": "Rare",
    "Sir": "Rare",
    "Jonkheer": "Rare",
    "Dona": "Rare"
  }
  out["Title"] = out["Title"].replace(title_map)
  out["Title"] = out["Title"].where(out["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")

  out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
  out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
  out["FamilySizeBand"] = pd.cut(out["FamilySize"], bins=[0, 1, 4, 20], labels=["Alone", "Small", "Large"])

  out["Deck"] = out["Cabin"].fillna("Unknown").astype(str).str[0]
  out["TicketPrefix"] = (
    out["Ticket"]
    .astype(str)
    .str.replace(r"[./]", " ", regex=True)
    .str.replace(r"\d+", "", regex=True)
    .str.strip()
    .replace("", "Unknown")
  )

  out["Embarked"] = out["Embarked"].fillna(out["Embarked"].mode()[0])
  out["Fare"] = out["Fare"].fillna(out["Fare"].median())
  out["FareLog"] = np.log1p(out["Fare"])

  return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)

target = "Survived"
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin", "AgeBand"]

x = train_fe.drop(columns=[target] + [c for c in drop_cols if c in train_fe.columns])
y = train_fe[target]
x_test = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

def fill_age(df):
  out = df.copy()
  out["Age"] = out["Age"].fillna(
    out.groupby(["Sex", "Pclass", "Title"])["Age"].transform("median")
  )
  out["Age"] = out["Age"].fillna(out["Age"].median())
  return out

x = fill_age(x)
x_test = fill_age(x_test)

cat_cols = x.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = x.select_dtypes(include=["number"]).columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

Categorical columns: ['Sex', 'Embarked', 'Title', 'FamilySizeBand', 'Deck', 'TicketPrefix']
Numerical columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone', 'FareLog']


In [ ]:
# Preprocessing + model comparison Submission 1: Score = 0.76794 (Logistic Regression vs Random Forest)

numeric_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
  ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
  ("num", numeric_pipe, num_cols),
  ("cat", categorical_pipe, cat_cols)
])

models = {
  "logreg": LogisticRegression(max_iter=2000, random_state=42),
  "rf": RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
  )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = {}

for name, model in models.items():
  pipe = Pipeline([("prep", preprocess), ("model", model)])
  cv_score = cross_val_score(pipe, x, y, cv=cv, scoring="accuracy", n_jobs=-1).mean()
  scores[name] = cv_score
  print(f"{name} CV Accuracy: {cv_score:.4f}")
  
best_model = max(scores, key=scores.get)
print(f"\nBest model: {best_model} with CV Accuracy: {scores[best_model]:.4f}")

logreg CV Accuracy: 0.8238
rf CV Accuracy: 0.8372

Best model: rf with CV Accuracy: 0.8372


In [ ]:
# Preprocessing + model comparison Submission 2: Score = 0.72727 (add ExtraTrees)
numeric_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
  ("num", numeric_pipe, num_cols),
  ("cat", categorical_pipe, cat_cols),
])

# [UPDATE FROM] submission 1: Add ExtraTreesClassifier to model comparison
model = ExtraTreesClassifier(
  n_estimators=1000,
  max_depth=None,
  min_samples_split=2,
  min_samples_leaf=1,
  random_state=42,
  n_jobs=-1
)

# [UPDATE FROM] submission 1: Use 10-fold stratified CV for more robust evaluation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
pipe = Pipeline([("prep", preprocess), ("model", model)])
cv_score = cross_val_score(pipe, x, y, cv=cv, scoring="accuracy", n_jobs=-1)

print(f"ExtraTrees CV Accuracy: {cv_score.mean():.4f}")

ExtraTrees CV Accuracy: 0.8047


In [50]:
# Preprocessing + model comparison Submission 3: Score = (Ensemble by RandomForest, ExtraTrees, Voting, Stacking)

numeric_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline([
  ("imputer", SimpleImputer(strategy="most_frequent")),
  ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
  ("num", numeric_pipe, num_cols),
  ("cat", categorical_pipe, cat_cols),
])

lr = LogisticRegression(max_iter=3000, random_state=42)

rf = RandomForestClassifier(
  n_estimators=1200,
  max_depth=8,
  min_samples_split=10,
  min_samples_leaf=3,
  max_features="sqrt",
  class_weight="balanced_subsample",
  random_state=42,
  n_jobs=-1
)

et = ExtraTreesClassifier(
  n_estimators=1200,
  max_depth=8,
  min_samples_split=10,
  min_samples_leaf=3,
  max_features="sqrt",
  class_weight="balanced_subsample",
  random_state=42,
  n_jobs=-1
)

voting_soft = VotingClassifier(
  estimators=[("lr", lr), ("rf", rf), ("et", et)],
  voting="soft",
  weights=[1, 2, 2],
  n_jobs=-1
)

stacking = StackingClassifier(
  estimators=[("rf", rf), ("et", et), ("lr", lr)],
  final_estimator=LogisticRegression(max_iter=3000, random_state=42),
  stack_method="predict_proba",
  cv=5,
  n_jobs=-1
)

models = {
  "rf": rf,
  "et": et,
  "voting_soft": voting_soft,
  "stacking": stacking
}

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = {}

for name, model in models.items():
  pipe = Pipeline([("prep", preprocess), ("model", model)])
  cv_score = cross_val_score(pipe, x, y, cv=cv, scoring="accuracy", n_jobs=-1).mean()
  scores[name] = cv_score
  print(f"{name} CV Accuracy: {cv_score:.5f}")

best_model = max(scores, key=scores.get)
print(f"\nBest model: {best_model} with CV Accuracy: {scores[best_model]:.5f}")

rf CV Accuracy: 0.82039
et CV Accuracy: 0.82599
voting_soft CV Accuracy: 0.82823
stacking CV Accuracy: 0.82262

Best model: voting_soft with CV Accuracy: 0.82823


In [ ]:
# Submission 1 had Random Forest as best model

best = models[best_model]
final_pipe = Pipeline([("prep", preprocess), ("model", best)])
final_pipe.fit(x, y)

predict = final_pipe.predict(x_test)

print(f"\nPredictions on test set (first 10): {predict[:10]}")

submission_df = pd.DataFrame({
  "PassengerId": test_df["PassengerId"],
  "Survived": predict.astype(int)
})

print("\nSubmission preview:")
print(submission_df.head())

submission_path = "titanic_submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"\nSubmission saved to {submission_path}")


Predictions on test set (first 10): [0 0 0 0 1 0 1 0 1 0]

Submission preview:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

Submission saved to titanic_submission.csv


In [44]:
# Submission 2 tried ExtraTreesClassifier

pipe.fit(x, y)
predict = pipe.predict(x_test)

submission_df = pd.DataFrame({
  "PassengerId": test_df["PassengerId"],
  "Survived": predict.astype(int)
})

submission_path = "titanic_submission_extratrees.csv"
submission_df.to_csv(submission_path, index=False)

print(f"\nSubmission with ExtraTrees saved to {submission_path}")


Submission with ExtraTrees saved to titanic_submission_extratrees.csv


In [51]:
# Submission 3 export using best ensemble model

best = models[best_model]
final_pipe = Pipeline([("prep", preprocess), ("model", best)])
final_pipe.fit(x, y)

predict = final_pipe.predict(x_test)

submission_df = pd.DataFrame({
  "PassengerId": test_df["PassengerId"],
  "Survived": predict.astype(int)
})

submission_path = f"titanic_submission_{best_model}.csv"
submission_df.to_csv(submission_path, index=False)

print("\nSubmission preview:")
print(submission_df.head())
print(f"\nSubmission saved to {submission_path}")


Submission preview:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1

Submission saved to titanic_submission_voting_soft.csv


In [45]:
files.download(submission_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Submission 1 had Random Forest as best model scoring 0.76794 
# Submission 2 tried ExtraTreesClassifier scoring 0.72727 
# Submission 3 tried ensemble methods (Voting and Stacking) scoring 0.77511.

# --- NOTE: The above code includes all recent edits. Do not suggest any code that has been deleted. ---

# --- RESULT: Based on the CV scores, Random Forest performed better than Logistic Regression and ExtraTreesClassifier. --- IGNORE ---
# --- RESULT: The ensemble methods (Voting and Stacking) provided a slight improvement over the individual Random Forest and ExtraTrees models, with Stacking being the best overall. --- IGNORE ---